# Adquisición y exploración inicial

**Objetivo:** Bajar los datos crudos, guardarlos en el disco local y realizar una primera inspección cuantitativa para conocer el contenido original de las fuentes antes de aplicar cualquier modificación o limpieza.

## 1. Arquitectura de Datos: Dataset Global vs. Dataset Local

Para lograr un modelo de detección de daños viales robusto y adaptable al Conurbano Bonaerense, la arquitectura de datos se compone de dos fuentes:

* **Dataset Global (RDD2022):** El *Road Damage Dataset 2022* es un estándar internacional masivo creado mediante *crowdsourcing* para competencias de visión computacional. Se utiliza para que la red neuronal aprenda las características universales y la topología general de los daños en el asfalto.
* **Dataset Local (Moreno):** Un conjunto de imágenes geolocalizadas en la Zona Oeste. Para construirlo, se extrajeron capturas crudas utilizando la API de *Mapillary* (una plataforma colaborativa de imágenes a nivel de calle tipo Street View), sumadas a grabaciones propias. Su propósito es adaptar el modelo global a la textura, iluminación y dispositivos de captura específicos de nuestro municipio.

## 2. Ingesta de Datos Crudos

A continuación, se utilizan las APIs correspondientes para descargar ambos conjuntos de datos. Los datos se almacenarán intactos en el directorio `data/raw/`.

> **Nota sobre credenciales:** Para replicar la descarga del dataset local, el entorno de ejecución debe contar con un archivo `.env` configurado. Es necesario poseer una cuenta gratuita en Roboflow y generar una API Key propia. El enlace a la documentación oficial se encuentra en la sección de Referencias al final de este documento.

## 3. Herramienta de Etiquetado


Se adoptó Roboflow como plataforma de operaciones de datos para el dataset local. Esto permitió estandarizar la curación manual de las cajas delimitadoras, aplicar transformaciones geométricas (Letterboxing) para evitar deformaciones, e inyectar robustez al modelo local mediante técnicas de aumento de datos (Data Augmentation).

## 4. Adquisición de Datos

In [26]:
import os
import urllib.request
import zipfile
from roboflow import Roboflow
from dotenv import load_dotenv, find_dotenv
import glob

In [ ]:
# Carga de variables de entorno locales
load_dotenv(find_dotenv())
api_key_robloflow = os.getenv('ROBOFLOW_API_KEY')

In [36]:
ruta_local = '../data/raw/dataset_local'
ruta_base_rdd = '../data/raw/RDD2022'
archivo_comprimido = '../data/raw/RDD2022.zip'
ruta_zips_internos = os.path.join(ruta_base_rdd, 'RDD2022_all_countries')

os.makedirs('../data/raw', exist_ok=True)

In [35]:
if not os.path.exists(ruta_local):
    print("Descargando Dataset Local (local)...")
    rf = Roboflow(api_key='eApsODD249C8ob9AQ9rG')
    project_local = rf.workspace("pics-workspace").project("deteccion-vial-moreno")
    dataset_local = project_local.version(1).download("yolo26", location=ruta_local)
else:
    print("Dataset Local ya existe, saltando descarga.")

Descargando Dataset Local (local)...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ../data/raw/dataset_local in yolo26:: 100%|██████████| 3811/3811 [00:02<00:00, 1678.91it/s]


In [10]:
url_directa_rdd = "https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/RDD2022.zip"
if not os.path.exists(ruta_base_rdd):
    print("\nIniciando descarga directa del Dataset Global (RDD2022)...")
    
    urllib.request.urlretrieve(url_directa_rdd, archivo_comprimido)
    print("Descarga completada. Descomprimiendo archivos...")
    
    with zipfile.ZipFile(archivo_comprimido, 'r') as zip_ref:
        zip_ref.extractall(ruta_base_rdd)
        
    print("Descompresion finalizada.")
    
    os.remove(archivo_comprimido)
    print("Archivo comprimido eliminado para ahorrar espacio en disco.")

else:
    print("Dataset Global ya existe, saltando descarga.")


Iniciando descarga directa del Dataset Global (RDD2022)...
Descarga completada. Descomprimiendo archivos...
Descompresion finalizada.
Archivo comprimido eliminado para ahorrar espacio en disco.


In [37]:
# Definicion estricta de filtros por Domain Shift
zips_a_ignorar = ['China_Drone.zip', 'United_States.zip']

print("--- Iniciando extraccion de subconjuntos geograficos ---")

if os.path.exists(ruta_zips_internos):
    for archivo in os.listdir(ruta_zips_internos):
        if archivo.endswith('.zip'):
            ruta_zip = os.path.join(ruta_zips_internos, archivo)
            
            # Aplicamos el filtro de Domain Shift
            if archivo in zips_a_ignorar:
                print(f"Descartado por Domain Shift: {archivo}")
                os.remove(ruta_zip) 
                continue
                
            nombre_pais = archivo.replace('.zip', '')
            ruta_extraccion_pais = os.path.join(ruta_base_rdd, nombre_pais)
            
            print(f"Extrayendo datos representativos de: {nombre_pais}...")
            os.makedirs(ruta_extraccion_pais, exist_ok=True)
            
            with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
                zip_ref.extractall(ruta_extraccion_pais)
                
            # Limpieza del zip interno para liberar espacio
            os.remove(ruta_zip)

    print("\nProceso de extraccion y purga finalizado.")
else:
    print("No se encontro el directorio de zips internos. Verifique la descarga principal.")

--- Iniciando extraccion de subconjuntos geograficos ---
Descartado por Domain Shift: China_Drone.zip
Extrayendo datos representativos de: China_MotorBike...
Extrayendo datos representativos de: Czech...
Extrayendo datos representativos de: India...
Extrayendo datos representativos de: Japan...
Extrayendo datos representativos de: Norway...
Descartado por Domain Shift: United_States.zip

Proceso de extraccion y purga finalizado.


## 3. Exploración Inicial Cruda

Se realiza un escaneo de los directorios recién descargados para evidenciar la estructura original de los datos suministrados por las fuentes, cuantificando el volumen de imágenes antes de la fase de integración.

In [40]:

print("--- Exploracion del Dataset Global Crudo (RDD2022) ---")

total_imagenes_rdd = 0
total_etiquetas_rdd = 0

# Listamos las carpetas extraidas ignorando archivos sueltos
carpetas_paises = [d for d in os.listdir(ruta_base_rdd) 
                   if os.path.isdir(os.path.join(ruta_base_rdd, d)) and d != 'RDD2022_all_countries']

for pais in carpetas_paises:
    ruta_pais = os.path.join(ruta_base_rdd, pais)
    
    # Conteo de imagenes (buscando en train y test recursivamente)
    imagenes = glob.glob(os.path.join(ruta_pais, '**', 'images', '*.jpg'), recursive=True)
    cant_img = len(imagenes)
    total_imagenes_rdd += cant_img
    
    # Conteo de etiquetas XML
    etiquetas = glob.glob(os.path.join(ruta_pais, '**', 'annotations', 'xmls', '*.xml'), recursive=True)
    cant_xml = len(etiquetas)
    total_etiquetas_rdd += cant_xml
    
    print(f" - {pais}: {cant_img} imagenes | {cant_xml} anotaciones XML")

print("-" * 40)
print(f"TOTAL IMAGENES RDD2022: {total_imagenes_rdd}")
print(f"TOTAL ETIQUETAS XML RDD2022: {total_etiquetas_rdd}")
print("-" * 40)
print("")

--- Exploracion del Dataset Global Crudo (RDD2022) ---
 - China_MotorBike: 2477 imagenes | 1977 anotaciones XML
 - Czech: 3538 imagenes | 2829 anotaciones XML
 - India: 9665 imagenes | 7706 anotaciones XML
 - Japan: 13133 imagenes | 10506 anotaciones XML
 - Norway: 10201 imagenes | 8161 anotaciones XML
----------------------------------------
TOTAL IMAGENES RDD2022: 39014
TOTAL ETIQUETAS XML RDD2022: 31179
----------------------------------------



> Nota: Se observa una disparidad entre imagenes y etiquetas (conjuntos Test no poseen XMLs).

In [42]:
print("--- Exploracion del Dataset Local Crudo (Moreno) ---")

total_imagenes_moreno = 0
total_etiquetas_moreno = 0

# Definicion de las particiones estandar de YOLO
splits_yolo = ['train', 'valid', 'test']

for split in splits_yolo:
    ruta_split = os.path.join(ruta_local, split)
    
    # Validacion de existencia de la particion
    if os.path.exists(ruta_split):
        # Conteo de imagenes JPG
        imagenes = glob.glob(os.path.join(ruta_split, 'images', '*.jpg'))
        cant_img = len(imagenes)
        total_imagenes_moreno += cant_img
        
        # Conteo de etiquetas de texto plano
        etiquetas = glob.glob(os.path.join(ruta_split, 'labels', '*.txt'))
        cant_txt = len(etiquetas)
        total_etiquetas_moreno += cant_txt
        
        print(f" - particion '{split}': {cant_img} imagenes | {cant_txt} anotaciones TXT")

print("-" * 40)
print(f"TOTAL IMAGENES MORENO: {total_imagenes_moreno}")
print(f"TOTAL ETIQUETAS TXT MORENO: {total_etiquetas_moreno}")
print("-" * 40)

--- Exploracion del Dataset Local Crudo (Moreno) ---
 - particion 'train': 1665 imagenes | 1665 anotaciones TXT
 - particion 'valid': 159 imagenes | 159 anotaciones TXT
 - particion 'test': 79 imagenes | 79 anotaciones TXT
----------------------------------------
TOTAL IMAGENES MORENO: 1903
TOTAL ETIQUETAS TXT MORENO: 1903
----------------------------------------


## 4. Definición Estricta del Alcance (Etiquetas)

Como se observa en la exploración, el dataset global incluye subconjuntos masivos (como drones en China o cámaras panorámicas de USA) y múltiples clases de daños menores que escapan al presupuesto de mantenimiento inmediato. 

Para justificar las acciones del próximo pipeline de integración (Notebook 02), se define el siguiente diccionario de etiquetas estricto que el modelo deberá aprender:

* **`D40` (Bache):** Falla estructural severa con pérdida de material.
* **`D20` (Piel de Cocodrilo):** Red de fisuras por fatiga de la base.
* **`calle_tierra`:** Etiqueta de contexto propia de la Zona Oeste para evitar falsos positivos en calles sin pavimentar.

En la próxima etapa, se filtrarán los datos crudos para alinearlos con este alcance y se eliminarán los subconjuntos mencionados.

---

## 5. Referencias y Enlaces Útiles

A continuación se adjuntan los enlaces de las fuentes y plataformas externas utilizadas para esta primer parte del trabajo.

* **Road Damage Dataset 2022 (RDD2022):** El conjunto de datos global crudo fue obtenido a través del repositorio estandarizado de Dataset Ninja.
  * Enlace: [RDD2022 en Dataset Ninja](https://datasetninja.com/road-damage-detector)
* **Mapillary:** Las imágenes locales fueron extraídas utilizando la Graph API para desarrolladores de esta plataforma.
  * Enlace: [Mapillary Developer API](https://www.mapillary.com/developer/api-documentation?locale=es_ES)
* **Roboflow:** Plataforma utilizada para la curación de las cajas delimitadoras, versionado y empaquetado del dataset local en formato YOLO.
  * Instrucciones para obtener credenciales: [Find Your Roboflow API Key](https://docs.roboflow.com/developer/authentication/find-your-roboflow-api-key)